# EDA

# Feature Engineering

This notebook prepares the dataset for the data modelling later
1. Clean the dataset
2. Add feature engineering

In [ ]:
# ==========================================
# STEP 1: LOAD & FIX DATA (The "Sanity Check")
# ==========================================
import pandas as pd
import numpy as np

# Load
df = pd.read_csv('synthetic_financial_data_NOISY (1).csv', sep='|')
df.columns = df.columns.str.strip()

# 1. REMOVE IMPOSSIBLE SCORES
# Credit Score must be between 300 and 850
print(f"Original Shape: {df.shape}")
df = df[(df['Credit_Score'] >= 300) & (df['Credit_Score'] <= 850)]
print(f"Shape after removing bad scores: {df.shape}")

# 2. REMOVE IMPOSSIBLE INCOMES
# Income must be reasonable (e.g., > 10k)
df = df[df['Annual_Income'] > 10000]

# 3. VERIFY CORRELATION
# Now check if the logic is fixed
print("\n--- New Correlation (Should be Negative) ---")
print(df[['Default_Status', 'Credit_Score']].corr())

# Save the fixed version
df.to_csv('final_clean_dataset_FIXED.csv', index=False, sep='|')
print("\n✅ Saved fixed dataset as 'final_clean_dataset_FIXED.csv'")

# Feature Engineering Latest

In [ ]:
import pandas as pd
from tqdm import tqdm
from google.colab import drive
from transformers import pipeline
import torch


#drive.mount('/content/drive')


file_path = 'final_clean_dataset_FIXED.csv'
df = pd.read_csv(file_path, sep='|')
print(f"Loaded {len(df)} rows.")


print("Running AI model")
device_id = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device=device_id)


occupation_labels = [
    "Medical", "Tech", "Education", "Business",
    "Trade", "Service", "Retired", "Student"
]

asset_labels = ["windfall or asset", "regular income"]


def ai_extraction(note):
    occ_result = classifier(str(note), occupation_labels, multi_label=False)
    best_occupation = occ_result['labels'][0]


    asset_result = classifier(str(note), asset_labels, multi_label=False)

    has_asset = "Yes" if asset_result['labels'][0] == "windfall or asset" else "No"

    return best_occupation, has_asset


print("Running AI Extraction")
tqdm.pandas()


df[['Occupation_Bucket', 'Has_External_Asset']] = df['Loan_Officer_Note'].progress_apply(
    lambda x: pd.Series(ai_extraction(x))
)


output_file = 'enriched_data_AI_ZeroShot.csv'
df.to_csv(output_file, sep='|', index=False)
print(f"✅ Done! Saved to {output_file}")


print("\nSample Results:")
print(df[['Loan_Officer_Note', 'Occupation_Bucket', 'Has_External_Asset']].head(10))

Decision Tree Model (Baseline)

In [ ]:
# Import libraries for Decision Tree baseline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [ ]:
# 1. Load the dataset
# Ensure the file name matches your local file
file_path = 'enriched_data_AI_ZeroShot.csv'
df = pd.read_csv(file_path, delimiter='|')

# 2. Preprocessing
# Drop columns that are identifiers or unstructured text (not suitable for a simple tree)
# We drop 'Customer_ID' and 'Loan_Officer_Note'.
X = df.drop(columns=['Customer_ID', 'Loan_Officer_Note', 'Default_Status'])
y = df['Default_Status']

# Convert 'Has_External_Asset' (Yes/No) to numerical (1/0)
le = LabelEncoder()
if 'Has_External_Asset' in X.columns:
    X['Has_External_Asset'] = le.fit_transform(X['Has_External_Asset'])

# Convert 'Occupation_Bucket' to numerical using One-Hot Encoding
X = pd.get_dummies(X, columns=['Occupation_Bucket'], drop_first=True)

# 3. Split the data into Training and Testing sets
# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Initialize and Train the Decision Tree Classifier
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# 5. Make Predictions and Evaluate
y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Optional: View Feature Importance
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': clf.feature_importances_
}).sort_values(by='Importance', ascending=False)

print("\nTop 5 Important Features:")
print(importances.head())